# Working with Data Providers

OceanStream supports multiple data providers, each designed to handle specific CSV formats from different oceanographic data sources.

## Built-in Providers
- **saildrone** - Saildrone USV data (CSV with `SD_` column prefixes)
- **r2r** - Rolling Deck to Repository (GeoCSV with metadata headers)

In [ ]:
# Setup
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

## List Available Providers

In [ ]:
from oceanstream import list_providers

providers = list_providers()
print("📋 Available providers:")
for name in providers:
    print(f"   • {name}")

## Getting a Provider

In [ ]:
from oceanstream import get_provider

# Get the Saildrone provider
saildrone = get_provider('saildrone')

print(f"Provider: {saildrone.name}")
print(f"Supported modules: {saildrone.supported_modules}")

In [ ]:
# Get the R2R provider (Research Vessels)
r2r = get_provider('r2r')

print(f"Provider: {r2r.name}")
print(f"Supported modules: {r2r.supported_modules}")

## Provider Capabilities

Each provider implements the `ProviderBase` protocol with these key methods:

In [ ]:
# Platform identification from filenames
test_files = [
    "sd1030_tpos_2023.csv",
    "sd1079_mission_data.csv", 
    "FK161229_607994_r2rnav.geocsv",
]

print("🔍 Platform detection:")
for filename in test_files:
    platform = saildrone.identify_platform(filename)
    if platform:
        print(f"   {filename} → Saildrone platform: {platform}")
    else:
        platform = r2r.identify_platform(filename)
        if platform:
            print(f"   {filename} → R2R cruise: {platform}")

In [ ]:
# Column alias mapping (provider-specific column names → standard names)
sample_columns = ['ship_longitude', 'ship_latitude', 'iso_time', 'speed_made_good']

aliases = r2r.alias_mapping(sample_columns)
print("📝 R2R column aliases:")
for orig, std in aliases.items():
    print(f"   {orig} → {std}")

## Using Providers with convert()

The provider parameter tells OceanStream how to interpret your data:

In [ ]:
from oceanstream import convert
import tempfile

# Setup paths
input_dir = project_root / "oceanstream" / "tests" / "data" / "raw_data"
output_dir = Path(tempfile.mkdtemp()) / "provider_demo"

# Use a specific provider
convert(
    provider=saildrone,  # Pass the provider object
    input_source=input_dir,
    output_dir=output_dir,
    campaign_id="provider_test",
    verbose=True,
    yes=True,
)

## Custom Providers

You can create custom providers by implementing the `ProviderBase` protocol:

```python
from oceanstream.providers.base import ProviderBase

class MyCustomProvider:
    name = "my_provider"
    supported_modules = ["geotrack"]
    
    def identify_platform(self, filename: str) -> str | None:
        if "myformat" in filename.lower():
            return filename.split("_")[0]
        return None
    
    def alias_mapping(self, columns):
        return {
            "my_lat": "latitude",
            "my_lon": "longitude",
            "my_time": "time",
        }
    
    def units_mapping(self, header, units_row=None):
        return {}
    
    def enrich_dataframe(self, df, metadata=None):
        return df
    
    def supports_module(self, module):
        return module in self.supported_modules

# Use it
convert(provider=MyCustomProvider(), ...)
```